In [ ]:
import re
import json
import os
import sys
import csv
from collections import defaultdict
from tqdm import tqdm
from drain3 import TemplateMiner

def pattern_to_lazy_regex(pattern):
    parts = re.split(r'(<\.\*>)', pattern)
    regex_parts = []
    for part in parts:
        if part == '<.*>':
            regex_parts.append('.*?')
        else:
            regex_parts.append(re.escape(part))
    return "".join(regex_parts)

def get_literals(pattern):
    return [seg for seg in re.split(r'<\.\*>', pattern) if seg]

def normalize_whitespace(s):
    return " ".join(s.split())

def evaluate_log_parsing_metrics(log_parsing_gt_data, pred_templates):
    """Evaluate log mining (log clustering level)"""
    print("🔍 Calculating log mining evaluation metrics...", file=sys.stderr)
    gt_patterns = set()
    for file_logs in log_parsing_gt_data.values():
        for entry in file_logs:
            gt_patterns.add(entry['template'].strip())
    gt_set = gt_patterns
    num_gt = len(gt_set)

    pred_patterns = set()
    for t in pred_templates:
        pattern = t.get('pattern', '').strip()
        if pattern:
            pred_patterns.add(pattern)
    num_pred = len(pred_patterns)

    correct = [p.strip() for p in tqdm(pred_patterns, desc="Comparing mining templates", unit="tmpl", file=sys.stderr) if p in gt_set]
    num_correct = len(correct)

    false_positive = sorted([p for p in pred_patterns if p not in gt_set])
    false_negative = sorted([p for p in gt_set if p not in pred_patterns])

    precision = num_correct / num_pred if num_pred > 0 else 0.0
    recall = num_correct / num_gt if num_gt > 0 else 0.0
    f1 = 2 * precision * recall / (precision + recall) if (precision + recall) > 0 else 0.0

    print("✅ Log mining evaluation completed!", file=sys.stderr)
    return {
        'precision': precision,
        'recall': recall,
        'f1': f1,
        'correct': correct,
        'false_positive': false_positive,
        'false_negative': false_negative,
        'num_correct': num_correct,
        'num_pred': num_pred,
        'num_gt': num_gt
    }

def generate_md_report(stats, output_path, eval_metrics=None, log_parsing_metrics=None, avg_tes=0.0):
    total_logs = stats['total_logs']
    matched_logs = stats['matched_logs']
    match_rate = (matched_logs / total_logs * 100) if total_logs > 0 else 0.0

    md_content = ["# Log Analysis Report\n\n"]

    if log_parsing_metrics is not None:
        md_content.append("## Log matched templates\n")
        md_content.append("| Metric | Value |\n")
        md_content.append("|------|-----|\n")
        md_content.append(f"| Precision | {log_parsing_metrics['precision']:.4f} ({log_parsing_metrics['num_correct']}/{log_parsing_metrics['num_pred']}) |\n")
        md_content.append(f"| Recall    | {log_parsing_metrics['recall']:.4f} ({log_parsing_metrics['num_correct']}/{log_parsing_metrics['num_gt']}) |\n")
        md_content.append(f"| F1-Score  | {log_parsing_metrics['f1']:.4f} |\n\n")

    md_content.append("## Statistical Overview\n")
    md_content.append(f"- Total log count: `{total_logs}`\n")
    md_content.append(f"- Matched log count: `{matched_logs}`\n")
    md_content.append(f"- Log match rate: `{match_rate:.2f}%`\n")
    md_content.append(f"- Number of matched templates: `{stats['matched_template_count']}`\n\n")

    md_content.append("## Template Matching Details\n")
    md_content.append("| Template Pattern | Match Count |\n")
    md_content.append("|----------|----------|\n")
    for pattern, count in stats['template_counts'].items():
        md_content.append(f"| `{pattern}` | {count} |\n")
    
    md_content.append("\n## Matched Content Details\n")
    for pattern, contents in stats['template_counts'].items():
        if pattern in stats['template_contents'] and stats['template_contents'][pattern]:
            md_content.append(f"### pattern `{pattern}`\n")
            contents = stats['template_contents'][pattern]
            md_content.append(f"**Match times**: {len(contents)}\n\n")
            md_content.append("```text\n")
            for idx, content in enumerate(contents[:3], 1):
                md_content.append(f"Match example {idx}:\n{content}\n{'─'*80}\n")
            md_content.append("```\n\n")

    md_content.append("\n## Unmatched Log Details\n")
    if stats['unmatched_logs']:
        md_content.append(f"Total {len(stats['unmatched_logs'])} unmatched logs\n\n")
        md_content.append("```text\n")
        for idx, content in enumerate(stats['unmatched_logs'][:50], 1):
            md_content.append(f"Unmatched log {idx}:\n{content}\n{'═'*80}\n")
        md_content.append("```\n")
    else:
        md_content.append("**All logs were successfully matched**\n")

    out_dir = os.path.dirname(output_path)
    if out_dir: os.makedirs(out_dir, exist_ok=True)
    with open(output_path, 'w', encoding='utf-8') as f:
        f.writelines(md_content)

LOG_PATTERN = re.compile(
    r'^(?P<timestamp>\d{4}-\d{2}-\d{2} \d{2}:\d{2}:\d{2},\d{3})\s+'
    r'(?P<level>ERROR|WARN|WARNING|INFO|DEBUG|TRACE)\s+'
    r'\[.*?\]\s*-\s*(?P<message>.+)$'
)
LEVEL_PATTERN = re.compile(r'\b(ERROR|WARN|WARNING|INFO|DEBUG|TRACE)\b')
CLASS_PREFIX_PATTERN = re.compile(
    r'^(?P<timestamp>\d{4}-\d{2}-\d{2} \d{2}:\d{2}:\d{2},\d{3})\s+'
    r'(?P<level>ERROR|WARN|WARNING|INFO|DEBUG|TRACE)\s+'
    r'\[[^\]]*\]\s+'
    r'(?P<class>[A-Za-z0-9_.$]+:\s+.*)$'
)

def extract_log_level_from_text(content):
    content_upper = content.upper()
    for level in ['ERROR', 'WARN', 'WARNING', 'INFO', 'DEBUG', 'TRACE']:
        if level in content_upper: return level
    return ''

def parse_log_line(line):
    match = LOG_PATTERN.match(line)
    if match:
        return {'raw': line.strip(), 'timestamp': match.group('timestamp'), 'level': match.group('level'), 'message': match.group('message').strip()}
    return None

def strip_class_prefix(msg: str) -> str:
    idx = msg.find(':')
    if idx == -1: return msg.strip()
    prefix = msg[:idx].strip()
    if prefix and re.match(r'^[A-Za-z0-9_.$]+$', prefix):
        return msg[idx + 1:].strip()
    return msg.strip()

def extract_message_body(raw_line: str) -> str:
    raw_line = raw_line.strip()
    msg = None
    sep = ' - '
    idx = raw_line.rfind(sep)
    if idx != -1: msg = raw_line[idx + len(sep):].strip()
    if msg is None:
        m = CLASS_PREFIX_PATTERN.match(raw_line)
        if m: msg = m.group('class').strip()
    if msg is None: msg = raw_line
    return strip_class_prefix(msg)

def normalize_log_entry(entry):
    if isinstance(entry, str):
        raw = entry.strip()
        parsed = parse_log_line(raw)
        if parsed: return {'raw': raw, 'level': parsed['level'], 'message': extract_message_body(parsed['message'])}
        return {'raw': raw, 'level': extract_log_level_from_text(raw), 'message': extract_message_body(raw)}
    elif isinstance(entry, dict):
        if 'content' in entry:
            content = str(entry['content']).strip()
            parsed = parse_log_line(content)
            if parsed: return {'raw': content, 'level': parsed['level'], 'message': extract_message_body(parsed['message'])}
            return {'raw': content, 'level': extract_log_level_from_text(content), 'message': extract_message_body(content)}
        elif 'message' in entry:
            msg = str(entry['message']).strip()
            lvl = str(entry.get('level', '')).strip().upper() or extract_log_level_from_text(msg)
            return {'raw': entry.get('raw', msg), 'level': lvl, 'message': extract_message_body(msg)}
    return {'raw': str(entry), 'level': extract_log_level_from_text(str(entry)), 'message': extract_message_body(str(entry))}

def strip_template_prefix(template: str) -> str:
    if not template: return template
    s = template.strip()
    idx = s.rfind(':')
    if idx == -1: return s
    prefix = s[:idx].strip()
    parts = prefix.split()
    last_token = parts[-1] if parts else ""
    if last_token and re.match(r'^[A-Za-z0-9_.$]+$', last_token):
        return s[idx + 1:].strip() or s
    return s

def mine_log_templates(log_data, output_file):
    template_miner = TemplateMiner()
    cluster_logs = defaultdict(list)
    cluster_templates = {}
    normalized_logs = [normalize_log_entry(e) for e in log_data]
    for log_entry in normalized_logs:
        result = template_miner.add_log_message(log_entry['message'])
        cid = result.get('cluster_id')
        if cid is not None: cluster_logs[cid].append(log_entry)
        if result.get('template_mined'): cluster_templates[cid] = result.get('template_mined')
    
    use_hdfs_optimization = 'HDFS' in output_file
    templates_info = []
    for cid, logs in sorted(cluster_logs.items()):
        template_str = cluster_templates.get(cid)
        if template_str:
            if use_hdfs_optimization: pattern = strip_template_prefix(template_str).replace('<*>', '<.*>')
            else: pattern = template_str.replace('<*>', '<.*>')
        else: pattern = ''
        levels = [log['level'] for log in logs if log['level']]
        level = max(set(levels), key=levels.count) if levels else ''
        templates_info.append({'pattern': pattern, 'level': level, 'method': '', 'occurrences': len(logs), 'representative_log': logs[0]['raw'] if logs else '', 'matched_logs': [log['raw'] for log in logs]})
    
    with open(output_file, 'w', encoding='utf-8') as f:
        json.dump(templates_info, f, ensure_ascii=False, indent=2)
    return templates_info

def match_logpai_logs(log_file, template_file, output_path, gt_file=None, **kwargs):
    anchor_min_len = kwargs.get('anchor_min_len', 3)
    max_sample_per_pattern = kwargs.get('max_sample_per_pattern', 100)
    
    logs = []
    with open(log_file, 'r', encoding='utf-8') as f:
        for line in f:
            if line.strip(): logs.append({'content': normalize_whitespace(line.strip())})
            
    with open(template_file, 'r', encoding='utf-8') as f:
        templates = json.load(f)
        
    compiled_templates = []
    for t in templates:
        pattern = normalize_whitespace(t['pattern'].replace('<*>', '<.*>'))
        compiled_templates.append({'pattern': pattern, 'regex': re.compile(pattern_to_lazy_regex(pattern)), 'is_plain': '<.*>' not in pattern, 'literals_lower': [lit.lower() for lit in get_literals(pattern) if len(lit) >= anchor_min_len]})
        
    stats = {'total_logs': len(logs), 'matched_logs': 0, 'matched_template_count': 0, 'template_counts': defaultdict(int), 'template_contents': defaultdict(list), 'unmatched_logs': []}
    log_match_count = {}

    for log_idx, log in enumerate(tqdm(logs, desc="Matching Logs")):
        content = log['content']
        content_lower = content.lower()
        matched_any = False
        matched_templates = []
        for tmpl in compiled_templates:
            if (tmpl['is_plain'] and tmpl['pattern'] in content) or (not tmpl['is_plain'] and (not tmpl['literals_lower'] or any(l in content_lower for l in tmpl['literals_lower'])) and tmpl['regex'].search(content)):
                pattern_key = tmpl['pattern']
                stats['template_counts'][pattern_key] += 1
                if len(stats['template_contents'][pattern_key]) < max_sample_per_pattern: stats['template_contents'][pattern_key].append(content)
                matched_any = True
                matched_templates.append(pattern_key)
        
        if matched_any: stats['matched_logs'] += 1
        else: stats['unmatched_logs'].append(content)
        log_match_count[f"log_{log_idx}"] = len(matched_templates) if matched_templates else 1

    stats['matched_template_count'] = len(stats['template_counts'])
    avg_tes = sum(1.0 / c for c in log_match_count.values()) / len(logs) if logs else 0
    
    log_parsing_metrics = None
    if gt_file:
        gt_data = {"logpai": []}
        with open(gt_file, 'r', encoding='utf-8') as f:
            reader = csv.DictReader(f)
            for row in reader:
                if 'EventTemplate' in row: gt_data["logpai"].append({'template': normalize_whitespace(row['EventTemplate'].replace('<*>', '<.*>'))})
        active_templates = [t for t in templates if normalize_whitespace(t['pattern'].replace('<*>', '<.*>')) in stats['template_counts']]
        log_parsing_metrics = evaluate_log_parsing_metrics(gt_data, [{'pattern': normalize_whitespace(t['pattern'].replace('<*>', '<.*>'))} for t in active_templates])

    generate_md_report(stats, output_path, log_parsing_metrics=log_parsing_metrics, avg_tes=avg_tes)
    return stats, log_parsing_metrics

def merge_and_rematch_for_logpai(original_template_file, original_stats, new_drain_templates, gt_file, output_report_path=None):
    with open(original_template_file, 'r', encoding='utf-8') as f:
        original_templates = json.load(f)
        
    matched_patterns_set = set(original_stats['template_counts'].keys())
    merged_templates = []
    seen = set()
    
    for t in original_templates:
        p = normalize_whitespace(t['pattern'].replace('<*>', '<.*>'))
        if p in matched_patterns_set and (p, t.get('level')) not in seen:
            merged_templates.append({'pattern': p, 'level': t.get('level')})
            seen.add((p, t.get('level')))
            
    for t in new_drain_templates:
        p = normalize_whitespace(t['pattern'].replace('<*>', '<.*>'))
        if (p, t.get('level')) not in seen:
            merged_templates.append({'pattern': p, 'level': t.get('level'), 'occurrences': t.get('occurrences', 0)})
            seen.add((p, t.get('level')))
            
    gt_data = {"logpai": []}
    with open(gt_file, 'r', encoding='utf-8') as f:
        reader = csv.DictReader(f)
        for row in reader:
            if 'EventTemplate' in row: gt_data["logpai"].append({'template': normalize_whitespace(row['EventTemplate'].replace('<*>', '<.*>'))})
            
    log_parsing_metrics = evaluate_log_parsing_metrics(gt_data, merged_templates)
    
    if output_report_path:
        template_counts = dict(original_stats['template_counts'])
        for t in merged_templates:
            p = t['pattern']
            if p not in template_counts: template_counts[p] = t.get('occurrences', 0)
            
        stats = {
            'total_logs': original_stats['total_logs'],
            'matched_logs': original_stats['matched_logs'] + sum(t.get('occurrences', 0) for t in new_drain_templates),
            'matched_template_count': len(merged_templates),
            'template_counts': template_counts,
            'template_contents': defaultdict(list),
            'unmatched_logs': []
        }
        generate_md_report(stats, output_report_path, log_parsing_metrics=log_parsing_metrics)
        
    return log_parsing_metrics

def eval_logpai_logs(model, dataset):
    stats, _ = match_logpai_logs(
        log_file=f'./dataset/{dataset}/{dataset}_2k.log',
        template_file=f'./log_template/{model}/{dataset}/processed_pattern.json',
        output_path=f'./report/{model}/{dataset}/white_box_only_report.md',
        gt_file=f'./dataset/{dataset}/{dataset}_2k.log_templates.csv'
    )
    drain_templates = mine_log_templates(log_data=stats['unmatched_logs'], output_file=f'./log_template/{model}/{dataset}/drain3_res.json')
    return merge_and_rematch_for_logpai(
        original_template_file=f'./log_template/{model}/{dataset}/processed_pattern.json',
        original_stats=stats,
        new_drain_templates=drain_templates,
        gt_file=f'./dataset/{dataset}/{dataset}_2k.log_templates.csv',
        output_report_path=f'./report/{model}/{dataset}/fused_report.md'
    )





In [ ]:
eval_logpai_logs('qwen3-32b', 'Zookeeper')